# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [1]:
# Import packages
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

import warnings

warnings.filterwarnings("ignore")

tf.keras.backend.set_floatx("float64")

# Define random seed for whole notebook
RSEED = 42

In [3]:
# Load data
df = pd.read_csv("C:\\Users\\dahan\\Desktop\\DataScience_neuefische_bootcamp_Berlin\\ds-artificial-neural-networksKSD\\data\\data\\boston.csv")

# Define target
y = df.pop("MEDV")

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

In [4]:
# Scale numerical features
# Scale numerical values
col_scale = [
    "CRIM",
    "ZN",
    "INDUS",
    "NOX",
    "RM",
    "AGE",
    "DIS",
    "TAX",
    "PTRATIO",
    "LSTAT",
]

scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [6]:
import os

print(os.getcwd())

c:\Users\dahan\Desktop\DataScience_neuefische_bootcamp_Berlin\ds-artificial-neural-networksKSD\day_2


In [8]:
print(os.listdir())

['-p', '03_overfit_underfit.ipynb', '04_load_saved_models.ipynb', 'saved_model', 'training_1']


In [ ]:
os.makedirs("saved_model", exist_ok=True)

In [11]:
!dir saved_model

 Volume in drive C has no label.
 Volume Serial Number is 5C61-FA53

 Directory of c:\Users\dahan\Desktop\DataScience_neuefische_bootcamp_Berlin\ds-artificial-neural-networksKSD\day_2\saved_model

16/09/2026  16:40    <DIR>          .
16/09/2026  16:40    <DIR>          ..
16/09/2026  16:40         9,577,377 my_large_model.keras
               1 File(s)      9,577,377 bytes
               2 Dir(s)  208,131,309,568 bytes free


In [15]:
from pathlib import Path

model_path = Path("saved_model/my_large_model.keras")

if model_path.exists():
    print("Found:", model_path)
    print("Size:", model_path.stat().st_size, "bytes")
else:
    print("File does not exist:", model_path)

Found: saved_model\my_large_model.keras
Size: 9577377 bytes


In [17]:
from tensorflow import keras

large_model = keras.models.load_model(
    "saved_model/my_large_model.keras"
)

print("Model loaded successfully!")

Model loaded successfully!


## SavedModel format

Reload a fresh Keras model from the saved model:

In [18]:
# Load the saved model
with tf.device("/cpu:0"):
    new_large_model = tf.keras.models.load_model("saved_model/my_large_model.keras")

# Check its architecture
new_large_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 512)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,385,412 (9.10 MB)

 Trainable params: 795,137 (3.03 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,590,275 (6.07 MB)

The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [19]:
# Evaluate the restored model
with tf.device("/cpu:0"):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f"Model MSE: {mse}")

4/4 - 1s - 159ms/step - loss: 7.3539 - mse: 90.7401
Model MSE: 90.74009967680405


The Model MSE is higher than in the notebook before, even though we are using the same data-split. We are using the same model and the same data. Therefore, we would assume that we also receive the same MSE. 

> **Exercise:** Can you find out where this notebook varies from the procedure in the notebook before? It might help to have a look at the values in X_train in both notebooks.

<details><summary>
Click here for a hint...
</summary>
Check out how the data were preprocessed. Did both notebooks use the same scaler?
</details>